# 01 — Data Splitting
Produces splits for two strategies: stratified (80/20 within-site) and hold-out time (2024–2025). Feature extraction happens downstream — no image loading here.

LOSO folds are derived inside `02b_features_loso.ipynb` directly from `dataset.csv` by partitioning on `AERONET_Site` — no pre-computed LOSO CSVs needed.

**Outputs:** 4 CSVs in `splits/`.

In [1]:
import pandas as pd
from datetime import datetime
from pathlib import Path

ROOT   = Path('D:/Regional-Scale-AOD-Estimation')
CSV    = ROOT / 'Data_Collection/Dataset/dataset.csv'
OUT    = ROOT / 'feature_engineering/splits'

df = pd.read_csv(CSV)
df = df.drop(columns=['Time(hh:mm:ss)', 'Day_of_Year(Fraction)'])

# Parse Date (dd:mm:yyyy) into a proper datetime for sorting
def parse_date(s):
    d, m, y = s.split(':')
    return datetime(int(y), int(m), int(d))

df['_date'] = df['Date(dd:mm:yyyy)'].apply(parse_date)
df = df.sort_values('_date').reset_index(drop=True)

print(f'Total samples : {len(df)}')
print(f'Sites         : {df["AERONET_Site"].nunique()}')
print(f'Date range    : {df["_date"].min().date()} → {df["_date"].max().date()}')

Total samples : 6183
Sites         : 20
Date range    : 2015-08-20 → 2025-11-30


## Split 1 — Stratified by site (chronological 80/20 within each site)

In [2]:
train_parts, val_parts = [], []

for site, grp in df.groupby('AERONET_Site'):
    grp = grp.sort_values('_date')
    cut = int(len(grp) * 0.8)
    train_parts.append(grp.iloc[:cut])
    val_parts.append(grp.iloc[cut:])

strat_train = pd.concat(train_parts).drop(columns='_date').reset_index(drop=True)
strat_val   = pd.concat(val_parts).drop(columns='_date').reset_index(drop=True)

strat_train.to_csv(OUT / 'stratified_train.csv', index=False)
strat_val.to_csv(OUT / 'stratified_val.csv', index=False)

print(f'Stratified  →  train: {len(strat_train)}  |  val: {len(strat_val)}')
print(f'Sites in train: {strat_train["AERONET_Site"].nunique()}  |  val: {strat_val["AERONET_Site"].nunique()}')

Stratified  →  train: 4940  |  val: 1243
Sites in train: 20  |  val: 20


## Split 2 — Hold-out time (train 2015–2023 / val 2024–2025)

In [4]:
time_train = df[df['_date'].dt.year <= 2023].drop(columns='_date').reset_index(drop=True)
time_val   = df[df['_date'].dt.year >= 2024].drop(columns='_date').reset_index(drop=True)

time_train.to_csv(OUT / 'holdout_time_train.csv', index=False)
time_val.to_csv(OUT / 'holdout_time_val.csv', index=False)

print(f'Hold-out time  →  train: {len(time_train)}  |  val: {len(time_val)}  ({100*len(time_val)/(len(time_train)+len(time_val)):.1f}%)')
print(f'Sites in val: {time_val["AERONET_Site"].nunique()}')

Hold-out time  →  train: 5166  |  val: 1017  (16.4%)
Sites in val: 11
